# Dynamic Text2SQL FastAPI on Colab

Runs the FastAPI app (Qwen2.5-3B + LoRA adapter, dynamic MySQL/PostgreSQL connection) on a Colab GPU runtime, exposed publicly via ngrok.

**Before running:**
1. Runtime -> Change runtime type -> GPU (T4 or better)
2. Have your ngrok authtoken ready (free at https://dashboard.ngrok.com/get-started/your-authtoken)
3. Have your checkpoint-3480 LoRA adapter available in Google Drive

In [ ]:
!nvidia-smi

## 1. Clone the repo

In [ ]:
REPO_URL = "https://github.com/PrathapShanmugam3/dynamic_text2sql_fastapi.git"

!rm -rf /content/app_repo
!git clone $REPO_URL /content/app_repo
%cd /content/app_repo

## 2. Install dependencies
Colab already has a matching CUDA torch build, so torch is skipped from requirements.txt to avoid a slow reinstall.

In [ ]:
!grep -v -i '^torch' requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt
!pip install -q pyngrok

## 3. Mount Google Drive (for the LoRA checkpoint)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configure environment variables
Set `ADAPTER_PATH` to wherever `checkpoint-3480` lives in your Drive.

In [ ]:
import os

os.environ["BASE_MODEL"] = "Qwen/Qwen2.5-3B-Instruct"
os.environ["ADAPTER_PATH"] = "/content/drive/MyDrive/texttosql/results/checkpoint-3480"  # <-- update this path
os.environ["MAX_NEW_TOKENS"] = "128"

assert os.path.exists(os.environ["ADAPTER_PATH"]), "ADAPTER_PATH not found -- fix the path above"

## 5. Configure ngrok and start the tunnel

In [ ]:
from google.colab import userdata
from pyngrok import ngrok, conf

# Preferred: store the token as a Colab secret named NGROK_AUTHTOKEN (key icon in left sidebar).
# Falls back to a manual prompt if the secret isn't set.
try:
    NGROK_AUTHTOKEN = '30zQ9fqxEvOM7L27fKWopqyzdYW_RvVhyqBudsTxyEyv7Y5o'
except Exception:
    NGROK_AUTHTOKEN = None

if not NGROK_AUTHTOKEN:
    import getpass
    NGROK_AUTHTOKEN = getpass.getpass("Enter your ngrok authtoken: ")

conf.get_default().auth_token = NGROK_AUTHTOKEN

## 6. Launch FastAPI (background) + open the tunnel

In [ ]:
import subprocess, time

server_process = subprocess.Popen(
    ["python3", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/app_repo",
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Waiting for model to load (this can take a few minutes on first run)...")
time.sleep(20)

In [ ]:
public_url = ngrok.connect(8000, "http")
print("Public API URL:", public_url)
print("Docs:", str(public_url) + "/docs")
print("Health check:", str(public_url) + "/health")

## 7. Tail server logs (optional, run to check status/errors)

In [ ]:
import time

for _ in range(50):
    line = server_process.stdout.readline()
    if not line:
        break
    print(line, end="")

## 8. Test the API

In [ ]:
import requests

resp = requests.get(f"{public_url}/health")
print(resp.status_code, resp.json())

In [ ]:
payload = {
    "database_type": "mysql",
    "host": "giftrewards-rupirewards.g.aivencloud.com",
    "port": 26043,
    "database": "defaultdb",
    "username": "avnadmin",
    "password": "YOUR_PASSWORD",
    "question": "Get all employees",
    "allow_limit": 1000
}

resp = requests.post(f"{public_url}/api/ask", json=payload)
print(resp.status_code)
print(resp.json())

## 9. Shutdown (run when done)

In [ ]:
ngrok.disconnect(public_url.public_url)
server_process.terminate()